In [2]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from caveat.encoding.continuous import ContinuousEncoder
from caveat.label_encoding import TokenAttributeEncoder
from caveat.mine_xy import DataModule, MutualInformationEstimator
from caveat.models.continuous.cvae_lstm import (
    ConcatEncoder,
    HiddenLabel,
    LabelEncoder,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

Device: cuda


/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [3]:
def latest(path: Path):
    versions = sorted(
        [
            d
            for d in path.iterdir()
            if d.is_dir() and d.name.startswith("version")
        ]
    )
    return Path(versions[-1])


def iter_models(path: Path):
    for dir in path.iterdir():
        if dir.is_dir():
            yield latest(dir)

In [4]:
label_encoders = {
    # "gender": TokenAttributeEncoder(config={"gender": "nominal"}),
    # "age_group": TokenAttributeEncoder(config={"age_group": "nominal"}),
    # "car_access": TokenAttributeEncoder(config={"car_access": "nominal"}),
    # "work_status": TokenAttributeEncoder(config={"work_status": "nominal"}),
    # "income": TokenAttributeEncoder(config={"income": "nominal"}),
    "all": TokenAttributeEncoder(
        config={
            "age": "nominal",
            "sex": "nominal",
            "employment": "nominal",
            "hh_income": "nominal",
            "hh_zone": "nominal",
            "day": "nominal",
            "vehicles": "nominal",
            "access_egress_distance": "nominal",
        }
    )
}
schedule_encoder = ContinuousEncoder()


def custom_loader(
    label_encoder,
    schedule_encoder,
    n: int = 5,
    shuffle_y: bool = False,
    zero_y: bool = False,
):
    ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))
    ys, _ = label_encoder.encode(ys)
    if shuffle_y:
        ys = ys[torch.randperm(ys.shape[0])]
    if zero_y:
        ys = ys * 0
    xs = pd.read_csv(Path("~/Data/foundata/out/nts/2023/activities.csv"))
    if "duration" not in xs.columns:
        xs["duration"] = xs.end - xs.start
    xs = schedule_encoder.encode(schedules=xs, labels=ys, label_weights=None)
    return xs

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Continuous Encoder initialised with:
        max_length: 12
        norm_duration: 1440
        jitter: 0
        fix_durations: stretch
        (act) weighting: unit
        (seq) joint weighting: unit
        trim eos: True
        


In [5]:
class MinerNet(nn.Module):
    def __init__(
        self,
        encodings,
        max_length,
        label_embed_sizes,
        hidden_size=256,
        encoder_depth=2,
        block_depth=2,
        dropout=0.2,
    ):
        super(MinerNet, self).__init__()

        self.label_embed = LabelEncoder(
            label_embed_sizes=label_embed_sizes, hidden_size=hidden_size
        )

        self.hidden = HiddenLabel(
            hidden_size=hidden_size,
            hidden_layers=encoder_depth,
            labels_size=hidden_size,
            dropout=dropout,
            activation=False,
        )

        self.schedule_encoder = ConcatEncoder(
            input_size=encodings,
            hidden_size=hidden_size,
            hidden_layers=encoder_depth,
            labels_size=hidden_size,
            max_length=max_length,
            dropout=dropout,
        )

        size = encoder_depth * hidden_size * 2

        blocks = []
        for _ in range(block_depth - 1):
            blocks.append(nn.Linear(size, hidden_size))
            if dropout > 0:
                blocks.append(nn.Dropout(dropout))
            blocks.append(nn.LeakyReLU())
            size = hidden_size
        self.blocks = nn.Sequential(*blocks, nn.Linear(size, 1))

    def forward(self, xs, ys):
        h0 = self.label_embed(ys.long())
        h1 = self.hidden(h0)
        h2 = self.schedule_encoder(xs, h0, h1)
        return self.blocks(h2)

In [16]:
hypers = {}
for alpha in [1]:
    results = {}
    for name, label_encoder in label_encoders.items():
        model_results = []

        for i in range(5):
            dataset = custom_loader(label_encoder, schedule_encoder)
            logger = TensorBoardLogger("logs/xy", name=f"{name}_{alpha}_{i}")
            loader = DataModule(
                dataset=dataset,
                val_split=0.1,
                test_split=0.1,
                batch_size=1024,
                num_workers=8,
                pin_memory=False,
            )

            net = MinerNet(
                encodings=dataset.activity_encodings,
                max_length=schedule_encoder.max_length,
                label_embed_sizes=label_encoder.label_kwargs[
                    "label_embed_sizes"
                ],
                hidden_size=256,
                encoder_depth=2,
                block_depth=2,
                dropout=0.2,
            )

            kwargs = {"alpha": alpha, "lr": 1e-3, "weight_decay": 1e-3}
            model = MutualInformationEstimator(net=net, **kwargs)
            trainer = Trainer(
                min_epochs=10,
                max_epochs=500,
                accelerator=device,
                devices=1,
                enable_progress_bar=False,
                logger=logger,
                enable_checkpointing=True,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=50),
                    ModelCheckpoint(
                        monitor="val_loss",
                        save_top_k=2,
                        save_weights_only=False,
                    ),
                ],
            )
            trainer.fit(model, datamodule=loader)
            mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
            model_results.append(mi)
        results[name] = {
            "mean": np.mean(model_results),
            "var": np.var(model_results),
        }
    hypers[alpha] = results

/tmp/ipykernel_2455468/203866817.py:30: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))


Encoding attribute 'age' with config: {'nominal': {'10-21': 0, '21-32': 1, '32-40': 2, '40-48': 3, '48-54': 4, '54-61': 5, '61-68': 6, '68-75': 7, '>75': 8, '≤10': 9}, 'location': 0, 'type': 'string'}
Data type after conversion: string
Data unique values: {'54-61', '61-68', '≤10', '21-32', '10-21', '40-48', '>75', '68-75', '32-40', '48-54'}
Existing encodings: {'10-21': 0, '21-32': 1, '32-40': 2, '40-48': 3, '48-54': 4, '54-61': 5, '61-68': 6, '68-75': 7, '>75': 8, '≤10': 9}


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.6 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.6 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


...

Encoding attribute 'access_egress_distance' with config: {'nominal': {'3.376-5.8': 0, '5.8-9.15862': 1, '9.15862-15.8': 2, '>15.8': 3, 'unknown': 4, '≤3.376': 5}, 'location': 0, 'type': 'string'}
Data type after conversion: string
Data unique values: {'5.8-9.15862', '≤3.376', '>15.8', 'unknown', '3.376-5.8', '9.15862-15.8'}
Existing encodings: {'3.376-5.8': 0, '5.8-9.15862': 1, '9.15862-15.8': 2, '>15.8': 3, 'unknown': 4, '≤3.376': 5}


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.6 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.6 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.04095908999443054    │
│          test_mi          │    0.04095908999443054    │
└───────────────────────────┴───────────────────────────┘

In [17]:
for depth, res in hypers.items():
    print(f"Depth {depth}:")
    for name, result in res.items():
        print(f"\tResults for {name}: {result}")

Depth 1:
	Results for age: {'mean': np.float64(0.32622442245483396), 'var': np.float64(0.00032810414273761525)}
	Results for hh_income: {'mean': np.float64(0.02025952860713005), 'var': np.float64(1.159815477600551e-05)}
	Results for sex: {'mean': np.float64(0.0061912685865536336), 'var': np.float64(1.262815790521498e-05)}
	Results for vehicles: {'mean': np.float64(0.0268690325319767), 'var': np.float64(1.97504691816619e-05)}
	Results for employment: {'mean': np.float64(0.37159401178359985), 'var': np.float64(0.00016787477848723142)}
	Results for day: {'mean': np.float64(0.03635658882558346), 'var': np.float64(3.401742707336286e-05)}
	Results for hh_zone: {'mean': np.float64(0.010060208849608898), 'var': np.float64(4.303837566628499e-06)}
	Results for access_egress_distance: {'mean': np.float64(0.042747630178928374), 'var': np.float64(4.956734324460532e-06)}


In [18]:
def joint_entropy(Y: np.ndarray) -> float:
    """H(y_1, y_2, ..., y_k) — entropy of the joint distribution.
    Treats each unique row as a single category."""
    _, counts = np.unique(Y, axis=0, return_counts=True)
    probs = counts / len(Y)
    return -np.sum(probs * np.log(probs))


ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))
cols = [
    "age",
    "hh_income",
    "sex",
    "vehicles",
    "employment",
    "day",
    "hh_zone",
    "access_egress_distance",
]
selected = ys[cols].to_numpy()
joint_h = joint_entropy(selected)
joint_h

/tmp/ipykernel_2455468/2169102580.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))


TypeError: The axis argument to unique is not supported for dtype object

In [ ]:
def entropy_discrete(c: np.ndarray) -> float:
    _, counts = np.unique(c, return_counts=True)
    probs = counts / len(c)
    return -np.sum(probs * np.log(probs))


ys = pd.read_csv(
    Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"), low_memory=False
)

entropies = {}
for col in [
    "age",
    "hh_income",
    "sex",
    "vehicles",
    "employment",
    "day",
    "hh_zone",
    "access_egress_distance",
]:
    data = ys[col].to_numpy()
    ent = entropy_discrete(data)
    print(f"Entropy of {col}: {ent:.4f}")
    entropies[col] = ent

Entropy of age: 2.3014
Entropy of hh_income: 1.6094
Entropy of sex: 0.6923
Entropy of vehicles: 1.2262
Entropy of employment: 1.5174
Entropy of day: 1.9458
Entropy of hh_zone: 1.0456
Entropy of access_egress_distance: 1.6663


In [19]:
cols = [
    "age",
    "hh_income",
    "sex",
    "vehicles",
    "employment",
    "day",
    "hh_zone",
    "access_egress_distance",
]
label_encoders = {n: TokenAttributeEncoder(config={n: "nominal"}) for n in cols}
hypers = {}
for alpha in [1]:
    results = {}
    for name, label_encoder in label_encoders.items():
        model_results = []

        for i in range(5):
            dataset = custom_loader(label_encoder, schedule_encoder)
            logger = TensorBoardLogger("logs/xy", name=f"{name}_{alpha}_{i}")
            loader = DataModule(
                dataset=dataset,
                val_split=0.1,
                test_split=0.1,
                batch_size=1024,
                num_workers=8,
                pin_memory=False,
            )

            net = MinerNet(
                encodings=dataset.activity_encodings,
                max_length=schedule_encoder.max_length,
                label_embed_sizes=label_encoder.label_kwargs[
                    "label_embed_sizes"
                ],
                hidden_size=256,
                encoder_depth=2,
                block_depth=2,
                dropout=0.2,
            )

            kwargs = {"alpha": alpha, "lr": 1e-3, "weight_decay": 1e-3}
            model = MutualInformationEstimator(net=net, **kwargs)
            trainer = Trainer(
                min_epochs=10,
                max_epochs=500,
                accelerator=device,
                devices=1,
                enable_progress_bar=False,
                logger=logger,
                enable_checkpointing=True,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=50),
                    ModelCheckpoint(
                        monitor="val_loss",
                        save_top_k=2,
                        save_weights_only=False,
                    ),
                ],
            )
            trainer.fit(model, datamodule=loader)
            mi = trainer.test(ckpt_path="best", datamodule=loader)[0]["test_mi"]
            model_results.append(mi)
        results[name] = {
            "mean": np.mean(model_results),
            "var": np.var(model_results),
        }
    hypers[alpha] = results

Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
            Label weighting: unit
            Joint weighting: unit
            
Label Encoder TokenAttributeEncoder initialised with:
 

/tmp/ipykernel_2455468/203866817.py:30: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ys = pd.read_csv(Path("~/Data/foundata/out/nts/2023/attributes_binned.csv"))


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.6 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.6 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


...

Encoding attribute 'access_egress_distance' with config: {'nominal': {'3.376-5.8': 0, '5.8-9.15862': 1, '9.15862-15.8': 2, '>15.8': 3, 'unknown': 4, '≤3.376': 5}, 'location': 0, 'type': 'string'}
Data type after conversion: string
Data unique values: {'5.8-9.15862', '≤3.376', '>15.8', 'unknown', '3.376-5.8', '9.15862-15.8'}
Existing encodings: {'3.376-5.8': 0, '5.8-9.15862': 1, '9.15862-15.8': 2, '>15.8': 3, 'unknown': 4, '≤3.376': 5}


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net         │ MinerNet │  1.6 M │ train │     0 │
│ 1 │ energy_loss │ Mine     │  1.6 M │ train │     0 │
└───┴─────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.6 M                                                                                                
Total estimated model params size (MB): 6                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/fred/miniforge3/envs/caveat/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (45) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │   -0.04685335233807564    │
│          test_mi          │    0.04685335233807564    │
└───────────────────────────┴───────────────────────────┘

In [20]:
for name, result in results.items():
    print(f"\tResults for {name}: {result}")

	Results for age: {'mean': np.float64(0.31611804366111756), 'var': np.float64(0.0004668860859938917)}
	Results for hh_income: {'mean': np.float64(0.015619532763957977), 'var': np.float64(9.021000374296664e-06)}
	Results for sex: {'mean': np.float64(0.00451779281720519), 'var': np.float64(1.2408361067677184e-06)}
	Results for vehicles: {'mean': np.float64(0.027832312136888505), 'var': np.float64(6.582384030580924e-06)}
	Results for employment: {'mean': np.float64(0.38318068385124204), 'var': np.float64(0.0005626829343744787)}
	Results for day: {'mean': np.float64(0.03586137667298317), 'var': np.float64(1.550778466264946e-05)}
	Results for hh_zone: {'mean': np.float64(0.009650052711367606), 'var': np.float64(6.323005374309649e-06)}
	Results for access_egress_distance: {'mean': np.float64(0.044932463020086286), 'var': np.float64(1.680132763941189e-05)}


In [21]:
normed = {}
for name, result in results.items():
    h = entropies[name]
    normed[name] = result["mean"] / h

pd.DataFrame(normed, index=["normalized_mi"]).T.sort_values(
    by="normalized_mi", ascending=False
)

,normalized_mi
employment,0.252533
age,0.137357
access_egress_distance,0.026965
vehicles,0.022698
day,0.018430
hh_income,0.009705
hh_zone,0.009229
sex,0.006526
